In [2]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
from tqdm import tqdm

import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

/Users/mehdisaurus/Documents/1Drittes/CV/jaguar project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Upload the Dataset
This script creates a FiftyOne dataset from the local screenshots directory. It scans for all image files in the screenshots folder and its subdirectories.

In [3]:
# Set up paths
image_dir = Path('../../data/intermediate/v1/screenshots')

# Create a new dataset
dataset_name = "jaguar_detection"
if fo.dataset_exists(dataset_name):
    print(f"Loading existing dataset: {dataset_name}")
    dataset = fo.load_dataset(dataset_name)
else:
    print(f"Creating new dataset: {dataset_name}")
    dataset = fo.Dataset(name=dataset_name, persistent=True)
    
    # Add all images from directory and subdirectories
    image_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        image_paths.extend(image_dir.rglob(ext))
    
    print(f"Found {len(image_paths)} images")
    
    samples = []
    for img_path in tqdm(image_paths, desc="Adding images"):
        sample = fo.Sample(filepath=str(img_path))
        samples.append(sample)
    
    dataset.add_samples(samples)
    print(f"Added {len(dataset)} samples to dataset")

Loading existing dataset: jaguar_detection


# Run Grounding-Dino
This script loads a Grounding DINO zero-shot object detection model from the FiftyOne model zoo, configured to detect either "jaguar's whole body" or "Close-up of a jaguar's head" based on the DETECTION_TYPE flag.   
It runs the model on the dataset, saving predictions in the appropriate raw_bboxes_body or raw_bboxes_head field, using a confidence threshold of 0.2 and a text similarity threshold of 0.6.  
It then selects the best detection depending on the chosen detection type and removes the raw bounding box field after processing.

### Test data set


In [10]:
# Create test dataset from test-detections folder
test_dataset_name = "jaguar_detection_test"
if fo.dataset_exists(test_dataset_name):
    print(f"Deleting existing test dataset: {test_dataset_name}")
    fo.delete_dataset(test_dataset_name)

print(f"Creating test dataset from test-detections folder")

# Set path to test images
test_image_dir = Path('../../data/intermediate/v1/screenshots/test-detections')

# Collect images from test directory
test_image_paths = []
for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
    test_image_paths.extend(test_image_dir.glob(ext))

print(f"Found {len(test_image_paths)} test images")

# Create test dataset
test_dataset = fo.Dataset(name=test_dataset_name, persistent=True)
samples = []
for img_path in test_image_paths:
    sample = fo.Sample(filepath=str(img_path))
    samples.append(sample)

test_dataset.add_samples(samples)
print(f"Test dataset created with {len(test_dataset)} samples")

Deleting existing test dataset: jaguar_detection_test
Creating test dataset from test-detections folder
Found 1 test images
 100% |█████████████████████| 1/1 [8.6ms elapsed, 0s remaining, 129.4 samples/s] 
Test dataset created with 1 samples
 100% |█████████████████████| 1/1 [8.6ms elapsed, 0s remaining, 129.4 samples/s] 
Test dataset created with 1 samples


In [11]:
# Configuration
DETECTION_TYPE = "body"  # or "head" - set this flag to choose processing type
USE_TEST_DATASET = True  # Set to False to use full dataset

# Select dataset
working_dataset = test_dataset if USE_TEST_DATASET else dataset
print(f"Working with {'test' if USE_TEST_DATASET else 'full'} dataset ({len(working_dataset)} samples)")

# Load appropriate model based on detection type
model = foz.load_zoo_model(
    "zero-shot-detection-transformer-torch",
    name_or_path="IDEA-Research/grounding-dino-tiny",
    classes=["jaguar's whole body" if DETECTION_TYPE == "body" else "Close-up of a jaguar's head"]
)

# Define the name of the bboxes field
raw_bboxes_name = f"raw_bboxes_{DETECTION_TYPE}"

# run model
working_dataset.apply_model(model,
                    label_field=raw_bboxes_name,
                    confidence_thresh=0.2)

print(f"✓ Model inference complete on {len(working_dataset)} samples")

Working with test dataset (1 samples)
Batch: 695bd78d71e6e5897399468f - 695bd78d71e6e5897399468f
Error: 'list' object has no attribute 'detach'
Traceback (most recent call last):
  File "/Users/mehdisaurus/Documents/1Drittes/CV/jaguar project/.venv/lib/python3.9/site-packages/fiftyone/core/models.py", line 466, in _apply_image_model_data_loader
    labels_batch = model.predict_all(imgs)
  File "/Users/mehdisaurus/Documents/1Drittes/CV/jaguar project/.venv/lib/python3.9/site-packages/fiftyone/utils/torch.py", line 886, in predict_all
    return self._predict_all(imgs)
  File "/Users/mehdisaurus/Documents/1Drittes/CV/jaguar project/.venv/lib/python3.9/site-packages/fiftyone/utils/transformers.py", line 976, in _predict_all
    return FiftyOneTransformer._predict_all(self, args)
  File "/Users/mehdisaurus/Documents/1Drittes/CV/jaguar project/.venv/lib/python3.9/site-packages/fiftyone/utils/transformers.py", line 683, in _predict_all
    return self._output_processor(
  File "/Users/mehdis

/Users/mehdisaurus/Documents/1Drittes/CV/jaguar project/.venv/lib/python3.9/site-packages/transformers/models/grounding_dino/processing_grounding_dino.py:93: FutureWarning: The key `labels` is will return integer ids in `GroundingDinoProcessor.post_process_grounded_object_detection` output since v4.51.0. Use `text_labels` instead to retrieve string object names.
  warnings.warn(self.message, FutureWarning)


 100% |█████████████████████| 1/1 [21.7s elapsed, 0s remaining, 0.0 samples/s] 
✓ Model inference complete on 1 samples
 100% |█████████████████████| 1/1 [21.7s elapsed, 0s remaining, 0.0 samples/s] 
✓ Model inference complete on 1 samples


In [14]:
# Helper functions for selecting best detection
def select_best_detection_body(dataset, raw_bboxes_field_name="raw_bboxes_body"):
    """Select detection with largest area as best body detection"""
    for sample in tqdm(dataset, desc="Selecting best body detection"):
        # Check if field exists and has detections
        if raw_bboxes_field_name not in sample or not sample[raw_bboxes_field_name]:
            continue
        
        raw_bboxes = sample[raw_bboxes_field_name]
        if raw_bboxes.detections:
            best_detection = max(
                raw_bboxes.detections,
                key=lambda d: d.bounding_box[2] * d.bounding_box[3],
            )
            sample["bboxes_body"] = fo.Detections(detections=[best_detection])
            sample.save()

def select_best_detection_head(dataset, raw_bboxes_field_name="raw_bboxes_head"):
    """Select detection with highest confidence as best head detection"""
    for sample in tqdm(dataset, desc="Selecting best head detection"):
        # Check if field exists and has detections
        if raw_bboxes_field_name not in sample or not sample[raw_bboxes_field_name]:
            continue
        
        raw_bboxes = sample[raw_bboxes_field_name]
        if raw_bboxes.detections:
            best_detection = max(
                raw_bboxes.detections,
                key=lambda d: d.confidence,
            )
            sample["bboxes_head"] = fo.Detections(detections=[best_detection])
            sample.save()

if DETECTION_TYPE == "body":
    # If you computed bboxes for whole body
    select_best_detection_body(working_dataset, raw_bboxes_field_name=raw_bboxes_name)
else:
    # If you computed bboxes for head only
    select_best_detection_head(working_dataset, raw_bboxes_field_name=raw_bboxes_name)

# Remove raw bboxes (only if field exists)
if raw_bboxes_name in working_dataset.get_field_schema():
    working_dataset.delete_sample_field(raw_bboxes_name)

print(f"✓ Best detection selection complete")

Selecting best body detection: 100%|██████████| 1/1 [00:00<00:00, 895.26it/s]

✓ Best detection selection complete


In [15]:
# Count detections in the working dataset
detection_field = "bboxes_body" if DETECTION_TYPE == "body" else "bboxes_head"
samples_with_detections = working_dataset.exists(detection_field)
total_samples = len(working_dataset)
detected_samples = len(samples_with_detections)

print(f"Detection Results:")
print(f"  Total samples: {total_samples}")
print(f"  Samples with detections: {detected_samples}")
print(f"  Samples without detections: {total_samples - detected_samples}")
print(f"  Detection rate: {detected_samples/total_samples*100:.1f}%")

Detection Results:
  Total samples: 1
  Samples with detections: 0
  Samples without detections: 1
  Detection rate: 0.0%


# Run SAM2
This code loads a SAM segmentation model and applies it to the dataset using bounding boxes as prompts.  
It stores the results in either segmentations_head or segmentations_body based on the detection type.

In [ ]:
# Define fields based on detection type
prompt_field = "bboxes_head" if DETECTION_TYPE == "head" else "bboxes_body"
label_field = "segmentations_head" if DETECTION_TYPE == "head" else "segmentations_body"

# Load the segmentation model
model = foz.load_zoo_model("segment-anything-vitb-torch")

# Apply the model to the dataset
dataset.apply_model(
    model,
    label_field=label_field,
    prompt_field=prompt_field,
)

# Store metadata locally

In [ ]:
storage_dir = Path('../../data/intermediate/v1/fo_dataset')
os.makedirs(storage_dir, exist_ok=True)

dataset.export(
    export_dir=str(storage_dir),
    dataset_type=fo.types.FiftyOneDataset,
    export_media=False,
    rel_dir=str(image_dir)
)